# STAR-v3 — Kaggle V3: text-LoRA · r=32 · PAIR-BATCH (hard-pair cua team data 100% in-batch)

**Cai dat:** GPU T4 · Internet ON · Add Input dataset nhu cu (tar.zst + xvlm_16m_base.th).

**3 cai tien thu nghiem (moi exp doi DUNG 1 thu so voi nen ~0.66 cua v1/v2):**
1. **v3a — MO LoRA TEXT** (`lora_freeze_text=false`): nghi pham so 1 cua plateau — query embedding do text
   encoder quyet dinh 100% ma 3 run truoc text bi dong bang hoan toan.
2. **v3b — text-LoRA + r=32/α=64**: them capacity (params LoRA ~x2, VRAM khong dang ke).
3. **v3c — PAIR-BATCH sampler** (`group_by=pair`): moi batch = 10 cap (anchor, hard-mine) tu 10 video
   KHAC nhau → 100% anchor luon co dung cap hard cua team data trong batch o MOI step (sampler cu chi
   dat ~1 lan/epoch/video), dong thoi khong lang phi slot vao positive cung video.
   *Da kiem tren data that: 8,647 cap train / 621 video, 0% cap trung sequence_id, batch 10/10 video khac nhau.*

**Moc so sanh:** v1 = 0.6615 · v2a = 0.6652 (hoa nhau; chenh < ~0.01 mAP = nhieu voi 621 query).
**Thoi gian uoc tinh:** v3a/v3b ~60–80'/run (tran 6 epoch + early-stop) · v3c ~1.2–2h (epoch pair dai ~1.7x,
tran 4 epoch) → tong ~3–4.5h. **VRAM:** text-LoRA them backward qua text encoder (~+0.5–1G tren nen 13.5G)
— gate cell do truoc; neu OOM ha batch 16.

In [ ]:
# [1/8] GPU check + clone/pull repo (pull lay PairBatchSampler moi)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
assert pathlib.Path("src/star/data/sampler.py").read_text().find("PairBatchSampler") > 0, "repo cu — pull lai!"
print("repo OK (co PairBatchSampler):", os.getcwd())

In [ ]:
# [2/8] Env + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/8] Tim dataset: checkpoint + data
import glob, os, pathlib, shutil

def find_one(pattern):
    hits = sorted(set(glob.glob(f"/kaggle/input/*/{pattern}") +
                      glob.glob(f"/kaggle/input/*/**/{pattern}", recursive=True)))
    return hits[0] if hits else None

CKPT  = find_one("xvlm_16m_base.th")
JSONL = find_one("train_10k_hard.jsonl")
ARCH  = find_one("train_10k_hard_data.tar.zst")

if JSONL:
    DATA_ROOT = str(pathlib.Path(JSONL).parents[2])
elif ARCH:
    marker = "/kaggle/working/extract"
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    if not got:
        os.makedirs(marker, exist_ok=True)
        print("extracting 1.6GB (~2-4 phut)...")
        if shutil.which("zstd"):
            os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
        else:
            import zstandard, tarfile
            with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf:
                    tf.extractall(marker)
        got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    assert got, "giai nen that bai"
    DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
else:
    raise FileNotFoundError("Khong thay data! Add Input dataset truoc.")

if not CKPT:
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(CKPT).stat().st_size > 8e8
print("DATA_ROOT =", DATA_ROOT)
print("CKPT      =", CKPT)

In [ ]:
# [4/8] Build manifest + ViTPose + PAIR LINK (da kiem tren data that: 11,735 rows, 0% cap trung seq)
import json, pathlib
import numpy as np, pandas as pd

root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")

for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line)
    anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"],
        pair_image_id=r.get("hard_i_id")))                  # <-- V3: link cap hard cua team data
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid, pair_image_id=None)

df = pd.DataFrame(anchors + [v for k, v in hard_rows.items() if k not in anchor_ids])
pose = json.load(open(root / "data/subsets/prepared/train_10k_hard_vitpose.json", encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid)
    b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of)
df["bbox"] = df.image_id.map(bbox_of)

rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""

MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"
df.to_parquet(MANIFEST, index=False)
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
n_pair = df[df.split == "train"].pair_image_id.notna().sum()
print(f"rows={len(df)} train={(df.split=='train').sum()} "
      f"valb_q={((df.split=='valb') & (df.caption!='')).sum()} valb_d={((df.split=='valb') & (df.caption=='')).sum()}")
print(f"kpts={df.keypoints.notna().mean():.0%} leakage={leak} anchor-co-pair(train)~{n_pair}")
assert leak == 0 and df.keypoints.notna().mean() > 0.99

In [ ]:
# [5/8] ====== BANG DIEU KHIEN THI NGHIEM V3 (moi exp doi 1 thu) ======
DATA = f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT}"

# Nen chung: nhu cau hinh tot da biet (LHP off, pose on, lam2=0.2, lr 2e-4, batch 20)
COMMON = ("data.lhp_enabled=false model.pose_enabled=true "
          "loss.lambda_smooth_ap=0.2 optim.lr_lora=2e-4 "
          "optim.epochs=6 train.early_stop_patience=6 train.eval_every_epochs=0.5 "
          "train.batch_size=20 train.grad_accum=2")

EXPERIMENTS = [
    {"name": "v3a_text_lora",     "set": "model.lora_freeze_text=false"},
    {"name": "v3b_text_lora_r32", "set": "model.lora_freeze_text=false model.lora_r=32 model.lora_alpha=64"},
    {"name": "v3c_pair_batch",    "set": "data.group_by=pair optim.epochs=4"},  # epoch pair dai ~1.7x
    # --- mo neu con gio ---
    # {"name": "v3d_combo",       "set": "model.lora_freeze_text=false model.lora_r=32 model.lora_alpha=64 data.group_by=pair optim.epochs=4"},
    # {"name": "v3e_pose_off",    "set": "model.pose_enabled=false"},
]
print(f"{len(EXPERIMENTS)} thi nghiem; COMMON: {COMMON}")

In [ ]:
# [6/8] GATE: overfit-one-batch o exp DAU TIEN (text-LoRA -> probe luon VRAM moi, xem dong vram=)
gate_cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch "
            f"--set {DATA} {COMMON} {EXPERIMENTS[0]['set']}")
print(gate_cmd)
!{gate_cmd}
print("Neu OOM: ha train.batch_size trong COMMON ve 16 roi chay lai tu cell nay.")

In [ ]:
# [7/8] CHAY TUAN TU
import pathlib, time, torch
RESULTS = []
for exp in EXPERIMENTS:
    out_dir = f"/kaggle/working/{exp['name']}"
    cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml "
           f"--set {DATA} {COMMON} {exp['set']} train.out_dir={out_dir}")
    print("=" * 100); print(">>>", exp["name"]); print(cmd)
    t0 = time.time()
    !{cmd}
    mins = round((time.time() - t0) / 60, 1)
    best, report = None, None
    ck = pathlib.Path(out_dir, "best.pth")
    if ck.exists():
        raw = torch.load(ck, map_location="cpu", weights_only=False)
        best = raw.get("best_metric")
        report = (raw.get("extra") or {}).get("report")
        del raw
    RESULTS.append(dict(name=exp["name"], mAP=best, mins=mins, report=report))
    print(f"<<< {exp['name']}: best mAP = {best}  ({mins} phut)")

In [ ]:
# [8/8] BANG SO SANH (moc: v1=0.6615, v2a=0.6652 — vung nhieu ±0.01)
import pandas as pd
tab = pd.DataFrame([{ "exp": r["name"], "best_mAP": r["mAP"], "phut": r["mins"],
                      "R@1": (r["report"] or {}).get("R@1"), "R@5": (r["report"] or {}).get("R@5"),
                      "R@10": (r["report"] or {}).get("R@10") } for r in RESULTS])
print(tab.sort_values("best_mAP", ascending=False).to_string())
print("\nMoc: v1=0.6615 (R@1 0.530) | v2a=0.6652 (R@1 0.536). Chenh < ~0.01 mAP = HOA.")
!ls -lh /kaggle/working/*/best.pth 2>/dev/null

## Doc ket qua
- Moi exp V3 doi **dung 1 bien** so voi nen ~0.66 → quy duoc cong/loi cho tung cai tien.
- **v3a thang ro (>+0.01)** → text frozen chinh la bottleneck → giu text-LoRA cho moi run sau
  (luu y: inference khi do text encoder CUNG da doi — khong dung checkpoint cu tron lan).
- **v3c thang ro** → pair-batch thanh mac dinh; dac biet ky vong ITM head manh hon nhieu
  (loss_itm se cao hon truoc — negative kho hon — do la dieu TOT, dung hoang).
- Hoa het → bi-encoder 10K da that su bao hoa → don luc sang re-rank + scale 50K/100K.

## Ghi chu ky thuat
- v3c: `PairBatchSampler` log so cap/batch ngay dau run (`8647 pairs -> 432 batches/epoch` voi batch 20).
- loss_itm trong v3c se KHONG tut nhanh ve ~0.02 nhu truoc — vi negative gio luon kho.
- VRAM: v3a/v3b co backward qua text encoder (+~0.5–1G tren nen 13.5G); gate cell da do truoc.
- Checkpoint tot nhat: `/kaggle/working/<exp>/best.pth` — ban giao theo `reports/instruct_inference.md`.